# Deep Learning Project — Experiments Notebook
2m Air Temperature Forecasting (ERA5, Delhi) with LSTM

This notebook walks through EDA, preprocessing, model training, the SOTA comparison, and the ablation study using the modules in `src/`.

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import load_raw, dataset_report, split_scale, create_sequences, TIME_STEPS
from model import build_baseline_lstm, build_attention_lstm


## 1. Load data & dataset report (Section 5A)

In [ ]:
df = load_raw('../data/2024-2026.csv')
report = dataset_report(df)
print('Samples:', report['n_samples'], '| Features:', report['n_features'])
print('Date range:', report['date_range'])
print('Missing values:', report['missing_values'])
print('Binary class balance (for classification metrics):', report['binary_class_balance'])
df.describe()

## 2. Exploratory plots

In [ ]:
plt.figure(figsize=(12,4))
df['Temp 2m'].plot()
plt.title('Temp 2m over time'); plt.ylabel('Kelvin'); plt.show()


In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df.drop(columns=['latitude','longitude']).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap'); plt.show()


## 3. Preprocess: chronological 70/15/15 split + scaling + sequences

In [ ]:
(X_train, X_val, X_test, y_train, y_val, y_test,
 x_scaler, y_scaler, split_info) = split_scale(df)
print(split_info)

X_train_seq, y_train_seq = create_sequences(X_train, y_train)
X_val_seq, y_val_seq = create_sequences(X_val, y_val)
X_test_seq, y_test_seq = create_sequences(X_test, y_test)
n_features = X_train_seq.shape[-1]
X_train_seq.shape, X_val_seq.shape, X_test_seq.shape

## 4. Train the baseline LSTM
See `src/train.py` for the full version with 5-fold CV. This is the quick, notebook-friendly version.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

model = build_baseline_lstm(n_features, TIME_STEPS)
history = model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=30, batch_size=128,
    callbacks=[EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
               ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)]
)

## 5. Evaluate on the held-out test set (regression + classification metrics)

In [ ]:
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              precision_score, recall_score, f1_score, accuracy_score,
                              confusion_matrix, roc_auc_score)

y_pred_scaled = model.predict(X_test_seq).flatten()
y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1,1)).flatten()
y_true = y_scaler.inverse_transform(y_test_seq.reshape(-1,1)).flatten()

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print(f'RMSE={rmse:.3f}  MAE={mae:.3f}  R2={r2:.4f}')

threshold = np.median(y_true)
y_true_c = (y_true >= threshold).astype(int)
y_pred_c = (y_pred >= threshold).astype(int)
cm = confusion_matrix(y_true_c, y_pred_c)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)
auc = roc_auc_score(y_true_c, y_pred)
print('Accuracy:', accuracy_score(y_true_c, y_pred_c))
print('Precision:', precision_score(y_true_c, y_pred_c))
print('Recall:', recall_score(y_true_c, y_pred_c))
print('F1:', f1_score(y_true_c, y_pred_c))
print('Specificity:', specificity)
print('ROC-AUC:', auc)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Below median','Above median'], yticklabels=['Below median','Above median'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix'); plt.show()

## 6. Training curves

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend(); plt.title('Loss vs Epoch'); plt.show()

## 7.Compare with baselines
Run the full comparison as a script (also trains Linear Regression, Dense MLP, GRU, Persistence on the same split):
```
!python ../src/compare_sota.py
```
Or inline below with just the persistence baseline as a quick sanity check:

In [ ]:
y_persist_scaled = np.roll(y_test.flatten(), 1)[TIME_STEPS:]
y_persist = y_scaler.inverse_transform(y_persist_scaled.reshape(-1,1)).flatten()
print('Persistence RMSE:', np.sqrt(mean_squared_error(y_true, y_persist)))
print('Proposed LSTM RMSE:', rmse)

## 8. Innovation: attention-augmented LSTM + ablation
Run the full ablation as a script:
```
!python ../src/ablation.py
```
Or train the attention model directly here:

In [ ]:
att_model = build_attention_lstm(n_features, TIME_STEPS, use_attention=True, use_bidirectional=True)
att_history = att_model.fit(
    X_train_seq, y_train_seq, validation_data=(X_val_seq, y_val_seq),
    epochs=30, batch_size=128,
    callbacks=[EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)]
)

att_pred = y_scaler.inverse_transform(att_model.predict(X_test_seq).reshape(-1,1)).flatten()
print('Attention+BiLSTM RMSE:', np.sqrt(mean_squared_error(y_true, att_pred)))
print('Baseline LSTM RMSE:   ', rmse)

## 9. Save final artifacts

In [ ]:
model.save('../models/baseline_model.keras')
att_model.save('../models/attention_model.keras')
pd.DataFrame(history.history).to_csv('../results/baseline_history.csv', index=False)
print('Saved.')